# RecAgent: демонстрация
84 вымышленных объекта, seed=42. Реальный API Recsflow не подключён. Выберите ядро Python из .venv проекта. Для запуска notebook нужен Jupyter/ipykernel (не требуется для чата и API).

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src' / 'recagent').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))
from recagent.agent import Agent
from recagent.models import ChatRequest
agent = Agent(mode='rules')
first = agent.chat(ChatRequest(message='Хочу лёгкий детективный сериал, один сезон', user_id='new-user'))
[(r.item.title, r.explanation) for r in first.recommendations]

In [ ]:
second = agent.chat(ChatRequest(message='Не дольше 30 минут', session_id=first.session_id, user_id='new-user'))
assert second.query.max_seasons == 1
assert second.recommendations
assert all(r.item.minutes <= 30 for r in second.recommendations)
second.model_dump()

In [ ]:
from recagent.grounding import validate_evidence
assert all(validate_evidence(e, r.item, second.query, []) for r in second.recommendations for e in r.evidence)
print('Каждый факт объяснения проверен по каталогу')

Для настоящего LLM-разбора создайте `Agent(mode='ollama')` при запущенной Ollama с qwen3:8b. Если модель недоступна, поле mode явно укажет rules_fallback. Сравните результаты с `report/evaluation-ollama.json`. Не трактуйте демонстрационные оценки как результат на реальных пользователях.